In [6]:
import fitz
import os
import shutil
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# ==========================================
# 階段一：資料處理與向量庫建立 (Data Ingestion)
# ==========================================

# 1. Load PDF (讀取 PDF)
doc = fitz.open("2025舞光LED21st(單頁水印可搜尋).pdf")
text = "".join([page.get_text() for page in doc])

# 2. Chunking (切分文本)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = text_splitter.split_text(text)

# 3. Embedding Model (設定本地端 EmbeddingGemma 模型)
# 確保你已在終端機執行過: ollama pull embeddinggemma
embeddings = OllamaEmbeddings(model="embeddinggemma")

# 4. 向量資料庫處理
persist_dir = "./test_gemma_vector_db"

# 每次執行都先清空舊的資料庫，確保資料是最新重切的
if os.path.exists(persist_dir):
    shutil.rmtree(persist_dir)
    print("已刪除舊的向量庫，準備以 EmbeddingGemma 模型重建...")

# 建立並儲存新的向量庫
vector_db = Chroma.from_texts(
    chunks, 
    embeddings, 
    persist_directory=persist_dir
)

print(f"成功！已建立資料庫。總共切分了 {len(chunks)} 個區塊。")

# ==========================================
# 階段二：檢索與大語言模型問答 (Retrieval & Generation)
# ==========================================

# 1. Setup Retriever (設定檢索器)
retriever = vector_db.as_retriever(search_kwargs={"k": 5})

# 2. Define the Local LLM (設定 Llama 4 Scout)
# 確保你已在終端機執行過: ollama pull llama4:scout
llm = ChatOllama(
    model="llama4:scout", 
    temperature=0  # 保持 0 讓回答精準，不隨便發揮創意
)

# 3. Create the Prompt (設定專家 Prompt)
template = """
### 角色定位
你是一位專業的「舞光 (DanceLight)」LED 照明產品顧問。你的任務是根據提供的目錄內容，為客戶提供準確、專業且親切的產品建議。

### 檢索到的目錄內容 (Context)
{context}

### 任務指令
1. **分析需求**：判斷使用者是尋找「特定產品名稱」（如：邱比特軌道燈）還是針對「特定場所」的需求（如：浴室、客廳）並且根據使用場景來推薦產品。
2. **規格導向**：在推薦產品時，請務必從 context 中提取以下技術屬性（若有標示）：
   - **瓦數 (W)**
   - **色溫 (K)**：如暖白光 (3000K)、自然光 (4000K) 或白光 (6000K)。
   - **演色性 (Ra)**：特別提到 Ra > 90 的高演色性產品。
   - **特殊功能**：如 IP65 防水等級、防眩光、低頻閃等。
3. **排版方式**：請使用「列點」格式，讓產品規格一目了然。

### 回答限制
- **範疇限制**：僅根據上方提供的「檢索到的目錄內容」進行回答。如果 context 中沒有相關資訊，請禮貌地回答：「抱歉，目前的產品目錄中沒有提到相關細節，建議您可以詢問其他類似款式。」
- **嚴禁胡說**：不可自行編造產品型號、規格或價格。
- **語言一致性**：一律使用「繁體中文」回答。

---
使用者問題：{question}
顧問建議：
"""
prompt = ChatPromptTemplate.from_template(template)

# 4. Build the Chain using LCEL (組裝 RAG 流程)
qa_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 5. Ask a question (提出問題)
print("\n[Llama 4 Scout 思考中...]\n")
response = qa_chain.invoke("天花板不需開槽的排燈是哪些型號")
print(response)

成功！已建立資料庫。總共切分了 441 個區塊。

[Llama 4 Scout 思考中...]

根據您提供的目錄內容，以下是一些不需開槽的天花板排燈選項：

1. **拉斐爾超薄磁吸軌道燈**：
   - 型號：D-UTMTTR8N、D-UTMTTR8W、D-UTMTTR15N、D-UTMTTR15W
   - 特點：超薄設計，磁吸安裝，無需開槽

2. **鋁槽燈系列**：
   - 型號：LED-1010ATWG、D-1010AT1-1BK、D-1010AT2-1BK、D-1010AT3-1BK
   - 特點：無縫安裝，適用於天花板

3. **軌道燈系列**：
   - 型號：LED-1212AT、LED-1507AT
   - 特點：安裝方便，無需開槽

請注意，以上產品的具體安裝方式和適用場景可能有所不同，建議您根據實際需求和場景選擇合適的產品。同時，您也可以參考產品目錄中的詳細介紹和技術參數，以確保選擇的產品符合您的需求。


In [10]:
# Use .stream() instead of .invoke()
for chunk in qa_chain.stream("舞蹈教室適合的 LED 燈有哪些？"):
    print(chunk, end="", flush=True)
print("\n")

舞蹈教室適合的LED燈需要考慮到照明的均勻性、色溫和演色性等因素。根據您提供的目錄內容，以下是一些適合舞蹈教室的LED燈產品：

1. **LED-TR30WFLR1**：
	* 功率：30W
	* 色溫：4000K（中性白光）
	* 流明：2050LM
	* 顯色指數：Ra≥90
2. **LED-TR30DFLBKR1**：
	* 功率：30W
	* 色溫：5000K（冷白光）
	* 流明：2400LM
	* 顯色指數：Ra≥80
3. **LED-2105R1**：
	* 功率：30W
	* 色溫：4000K（中性白光）
	* 流明：2050LM
	* 顯色指數：Ra≥90

這些產品都具有高顯色指數（Ra≥80或≥90），能夠提供均勻且高質量的照明。同時，它們的色溫也適合舞蹈教室的需求，能夠提供舒適且自然的光線。

此外，您也可以考慮以下因素來選擇適合的LED燈：

* 照明面積：根據舞蹈教室的大小，選擇合適的LED燈數量和佈置方案。
* 調光功能：考慮是否需要調光功能，以滿足不同的照明需求。
* 防水等級：如果舞蹈教室存在濕度較高的區域，建議選擇具有防水等級的LED燈。

建議您根據實際需求和預算，選擇適合的LED燈產品。同時，也可以咨詢專業人士或廠商，獲取更詳細的建議和方案。



In [13]:
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Load EXISTING Vector DB (No PDF parsing here!)
embeddings = OllamaEmbeddings(model="embeddinggemma")
vector_db = Chroma(
    persist_directory="./test_gemma_vector_db", 
    embedding_function=embeddings
)

retriever = vector_db.as_retriever(search_kwargs={"k": 5})

# 2. Define LLM (Ensure GPU acceleration is active)
llm = ChatOllama(
    model="llama4:scout", 
    temperature=0,
    num_gpu=99,      # Forces Ollama to put all layers into VRAM
    keep_alive="30m"  # Keeps the model in VRAM indefinitely
)

# 3. Create the Prompt (Same as your original)
template = """
### 角色定位
你是一位專業的「舞光 (DanceLight)」LED 照明產品顧問。你的任務是根據提供的目錄內容，為客戶提供準確、專業且親切的產品建議。

### 檢索到的目錄內容 (Context)
{context}

### 任務指令
1. **分析需求**：判斷使用者是尋找「特定產品名稱」還是針對「特定場所」的需求，並根據使用場景來推薦產品。
2. **規格導向**：在推薦產品時，請提取以下屬性（若有標示）：瓦數 (W)、色溫 (K)、演色性 (Ra)、特殊功能。
3. **排版方式**：請使用「列點」格式。

### 回答限制
- 僅根據上方提供的「檢索到的目錄內容」進行回答。
- 嚴禁自行編造產品型號、規格或價格。
- 一律使用「繁體中文」回答。

---
使用者問題：{question}
顧問建議：
"""
prompt = ChatPromptTemplate.from_template(template)

# 4. Build RAG Chain
qa_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 5. Execute with STREAMING
print("\n[Llama 4 Scout 思考中...]\n")
question = "舞蹈教室適合的 LED 燈有哪些？"

# Use .stream() instead of .invoke()
for chunk in qa_chain.stream(question):
    print(chunk, end="", flush=True)
print("\n")


[Llama 4 Scout 思考中...]

舞蹈教室適合的LED燈需要考慮到照明的均勻性、色溫和演色性等因素。根據目錄內容，以下是一些建議：

1. **LED-CE30WR3**：
	* 30W
	* 3000K（暖白光）
	* 2050LM
	* 時尚白
2. **LED-CE30DFR3**：
	* 30W
	* 5000K（冷白光）
	* 2400LM
	* 時尚白

這些產品具有高演色性（Ra≥80），適合舞蹈教室的照明需求。同時，LED燈具有節能、壽命長等優點。

此外，還可以考慮以下因素：

* **色溫**：舞蹈教室通常需要暖白光或冷白光，具體取決於個人喜好和舞蹈類型。
* **演色性**：舞蹈教室需要高演色性（Ra≥80）以確保燈光的真實性和舒適性。
* **亮度**：舞蹈教室需要適當的亮度，以確保燈光不會過於刺眼或不足。

綜合考慮，建議選擇**LED-CE30WR3**或**LED-CE30DFR3**，具體取決於個人喜好和舞蹈教室的需求。

